In [23]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current",
]

Note: you may need to restart the kernel to use updated packages.


In [24]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [25]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间与电流处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        current_values = group["Current"]

        # 如果首个测量点 Current == 0，则忽略首点
        if len(group) >= 2 and group["Current"].iloc[0] == 0:
            current_values = current_values.iloc[1:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    pulse_sequence = pulse_sequence.groupby(["File", "pulse_segment_id"]).filter(
        lambda group: not is_bad_current_segment(group)
    ).copy()

    # -----------------------------
    # 7. 每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def pick_pulse_row(group):
        group = group.sort_values("Time")

        # 如果该脉冲段第一个测量点 Current == 0，
        # 就改用第二个测量点记录
        if group["Current"].iloc[0] == 0 and len(group) >= 2:
            return group.iloc[1]

        return group.iloc[0]

    pulse_sequence = (
        pulse_sequence
        .groupby(["File", "pulse_segment_id"], group_keys=False)
        .apply(pick_pulse_row)
        .reset_index(drop=True)
    )

    # -----------------------------
    # 8. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence


pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

display(pulse_sequence)


C:\Users\12639\AppData\Local\Temp\ipykernel_4824\1410155938.py:133: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_pulse_row)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.500000+00:00,-1.498660,4.038494,DCH,12_17,DCH/-1.5
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,4.121693,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,4.017813,DCH,12_25,DCH/-3.0
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.400000+00:00,-1.497221,3.726728,DCH,12_35,DCH/-1.5
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,3.804263,CHA,12_39,CHA/1.5
...,...,...,...,...,...,...,...,...,...
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,3.829836,CHA,9_48,CHA/3.0
258,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.550000+00:00,-1.494522,3.194370,DCH,9_54,DCH/-1.5
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,3.331177,CHA,9_58,CHA/1.5
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,3.244738,DCH,9_62,DCH/-3.0


In [26]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
# -----------------------------
def filter_pulse(df):

    pulse_sequence_filter = df[~df["Zustand/Current"].isin(["DCH/-1.5"])].copy()
    return pulse_sequence_filter

filtered_pulse = filter_pulse(pulse_sequence)
display(filtered_pulse)

,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,4.121693,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,4.017813,DCH,12_25,DCH/-3.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,3.804263,CHA,12_39,CHA/1.5
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.020000+00:00,-2.997524,3.721058,DCH,12_43,DCH/-3.0
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.330000+00:00,2.992477,3.833616,CHA,12_47,CHA/3.0
...,...,...,...,...,...,...,...,...,...
256,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.320000+00:00,-2.998784,3.718723,DCH,9_44,DCH/-3.0
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,3.829836,CHA,9_48,CHA/3.0
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,3.331177,CHA,9_58,CHA/1.5
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,3.244738,DCH,9_62,DCH/-3.0


In [27]:
# -----------------------------
# 10. 为当前 filter 后的 pulse 匹配前一个 PAUO 的最后时刻数据
# 输出：
#   1) filtered_pauo：每个 filtered_pulse 前序的最后一个 PAUO 点
#   2) pauo_pulse_table：筛选出的 PAUO 与 filtered_pulse 合并表
# -----------------------------

def build_pauo_pulse_tables(time_diff_sequence, filtered_pulse):
    # 只从完整 time_diff_sequence 里取 PAUO，这样 PAUO 也保留 SOH / SOC / File 等列
    pauo_points = time_diff_sequence[time_diff_sequence["Zustand"].eq("PAUO")].copy()

    pauo_points["pause_end_time"] = pd.to_datetime(pauo_points["Time"], utc=True, errors="coerce").astype("datetime64[ns, UTC]")

    pauo_points = (pauo_points.dropna(subset=["File", "pause_end_time"]).reset_index(drop=True))
    pauo_points["pauo_row"] = np.arange(len(pauo_points))

    pulse_for_match = filtered_pulse.copy().reset_index(drop=True)
    pulse_for_match["pulse_time"] = pd.to_datetime(pulse_for_match["Time"], utc=True, errors="coerce").astype("datetime64[ns, UTC]")
    pulse_for_match["pulse_order"] = np.arange(len(pulse_for_match))

    valid_pulse = pulse_for_match.dropna(subset=["File", "pulse_time"]).copy()

    # 对每一个 filtered_pulse，向前找同一个 File 内时间最近的 PAUO 点
    pause_end_match = pd.merge_asof(
        valid_pulse[["File", "pulse_time", "pulse_order"]]
        .sort_values(["pulse_time", "File"]),

        pauo_points[["File", "pause_end_time", "pauo_row"]]
        .sort_values(["pause_end_time", "File"]),
        left_on="pulse_time",
        right_on="pause_end_time",
        by="File",
        direction="backward"
    )

    pause_end_match = (
        pause_end_match
        .sort_values("pulse_order")
        .reset_index(drop=True)
    )
    pause_end_match["match_id"] = np.arange(1, len(pause_end_match) + 1)

    # 表 1：筛选出的 PAUO 点
    matched_pauo_pairs = pause_end_match.dropna(subset=["pauo_row"])[
        ["pulse_order", "pauo_row", "match_id"]
    ].copy()
    
    filtered_pauo = (
        matched_pauo_pairs
        .merge(pauo_points, on="pauo_row", how="left")
        .drop(columns=["pulse_order", "pauo_row", "pause_end_time"], errors="ignore")
        .reset_index(drop=True)
    )
    filtered_pauo = filtered_pauo[["match_id"] + OUTPUT_COLUMNS ].copy()

    # 表 2：PAUO + PULSE 合并表
    pauo_output = filtered_pauo.copy()
    
    pulse_output = (
        pulse_for_match
        .merge(
            pause_end_match[["pulse_order", "pauo_row", "match_id"]],
            on="pulse_order",
            how="left"
        )
        .drop(columns=["pulse_time", "pulse_order", "pauo_row"], errors="ignore")
    )
    

    common_columns = ["match_id"] + OUTPUT_COLUMNS 

    pauo_pulse_table = pd.concat(
        [
            pauo_output[common_columns],
            pulse_output[common_columns]
        ],
        ignore_index=True,
        sort=False,
    )

    pauo_pulse_table = (
    pauo_pulse_table
    .sort_values(["File", "match_id", "Time"], na_position="last")
    .drop(columns=["match_id"])
    .reset_index(drop=True)
    )

# 单独从表1删除 match_id
    filtered_pauo = (
    filtered_pauo
    .drop(columns=["match_id"])
    .reset_index(drop=True)
    )
    
    return filtered_pauo, pauo_pulse_table

filtered_pauo, pauo_pulse_table = build_pauo_pulse_tables(
    time_diff_sequence,
    filtered_pulse,
)

display(filtered_pauo)
display(pauo_pulse_table)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:34.750000+00:00,0.0,4.087893,PAUO,12_20,PAUO/0.0
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:17.890000+00:00,0.0,4.088227,PAUO,12_24,PAUO/0.0
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:38.910000+00:00,0.0,3.776801,PAUO,12_38,PAUO/0.0
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:22.170000+00:00,0.0,3.777579,PAUO,12_42,PAUO/0.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:25.540000+00:00,0.0,3.779025,PAUO,12_46,PAUO/0.0
...,...,...,...,...,...,...,...,...,...
189,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:48.480000+00:00,0.0,3.773688,PAUO,9_43,PAUO/0.0
190,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:51.800000+00:00,0.0,3.775133,PAUO,9_47,PAUO/0.0
191,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:30.920000+00:00,0.0,3.300823,PAUO,9_57,PAUO/0.0
192,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.100000+00:00,0.0,3.305715,PAUO,9_61,PAUO/0.0


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:34.750000+00:00,0.000000,4.087893,PAUO,12_20,PAUO/0.0
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,4.121693,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:17.890000+00:00,0.000000,4.088227,PAUO,12_24,PAUO/0.0
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,4.017813,DCH,12_25,DCH/-3.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:38.910000+00:00,0.000000,3.776801,PAUO,12_38,PAUO/0.0
...,...,...,...,...,...,...,...,...,...
383,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,3.331177,CHA,9_58,CHA/1.5
384,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.100000+00:00,0.000000,3.305715,PAUO,9_61,PAUO/0.0
385,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,3.244738,DCH,9_62,DCH/-3.0
386,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 07:25:17.580000+00:00,0.000000,3.317167,PAUO,9_65,PAUO/0.0


In [28]:
# -----------------------------
# 11. 计算 R0 = delta_Voltage / delta_Current
# -----------------------------
def resistance_r0(pauo_pulse_table):
    r0_table = (
    pauo_pulse_table
    .sort_values(["File", "Time"])
    .reset_index(drop=True)
    .copy()
    )

    id_parts = r0_table["ID"].astype(str).str.rsplit("_", n=1, expand=True)
    
    r0_table["id_number"] = pd.to_numeric(
    id_parts[1],
    errors="coerce"
    )

    id_is_adjacent = (r0_table["File"].eq(r0_table["File"].shift(1))

    & r0_table["id_number"].eq(r0_table["id_number"].shift(1) + 1)
    )

    r0_table["delta_Current"] = (
    r0_table["Current"]- r0_table["Current"].shift(1)
    )

    r0_table["delta_Voltage"] = (
    r0_table["Voltage"]- r0_table["Voltage"].shift(1)
    )
    
    r0_table["R0"] = (r0_table["delta_Voltage"]/ r0_table["delta_Current"] * 1000).abs().where(id_is_adjacent)

    r0_table = (r0_table[r0_table["Zustand"].ne("PAUO")].reset_index(drop=True))

    r0_result = (
    r0_table.drop(columns=["id_number", "delta_Current", "delta_Voltage"])
    .reset_index(drop=True)
    )

    return r0_result

r0_result = resistance_r0(pauo_pulse_table)
display(r0_result)

,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,4.121693,CHA,12_21,CHA/1.5,22.560928
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,4.017813,DCH,12_25,DCH/-3.0,23.507635
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,3.804263,CHA,12_39,CHA/1.5,18.318566
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.020000+00:00,-2.997524,3.721058,DCH,12_43,DCH/-3.0,18.856021
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.330000+00:00,2.992477,3.833616,CHA,12_47,CHA/3.0,18.242787
...,...,...,...,...,...,...,...,...,...,...
189,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.320000+00:00,-2.998784,3.718723,DCH,9_44,DCH/-3.0,18.329025
190,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,3.829836,CHA,9_48,CHA/3.0,18.239417
191,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,3.331177,CHA,9_58,CHA/1.5,20.284600
192,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,3.244738,DCH,9_62,DCH/-3.0,20.406411


In [43]:
# =============================================================================
# 12. 二阶 RC 拟合：在已计算 R0 的基础上计算 R1 / R2
# =============================================================================
# 模型说明：
#   这里沿用原始数据的电流符号：CHA 为正，DCH 为负。
#   V = OCV + Current * R0 + V1 + V2
#   dV1/dt = -V1/tau1 + Current*R1/tau1
#   dV2/dt = -V2/tau2 + Current*R2/tau2
#
# 拟合窗口：
#   前一个 PAUO 末端点用于初始化 OCV 和 RC 状态；
#   当前 pulse 段的所有采样点参与拟合；
#   当前 pulse 后面的 PAUO 段所有采样点参与拟合。
#
# 输出：
#   R0 / R1 / R2 单位均为 mOhm
#   tau1 / tau2 单位为 s
#   C1 / C2 单位为 F
# =============================================================================


def resistance_r1r2(time_diff_sequence, r0_result):

    seq = time_diff_sequence.copy()
    seq["Zustand_clean"] = (seq["Zustand"].str.strip().str.replace(r"\*+$", "", regex=True))
    seq["Time_dt"] = pd.to_datetime(seq["Time"], utc=True, errors="coerce")

    seq = (
        seq
        .dropna(subset=["Time_dt", "Current", "Voltage"])
        .sort_values(["File", "Time_dt"])
        .reset_index(drop=True)
    )

    # 重新根据 Zustand 连续性划分片段，用来找完整 pulse 段和后续 PAUO 段
    seq["segment_id"] = (seq["File"].ne(seq["File"].shift()) | seq["Zustand_clean"].ne(seq["Zustand_clean"].shift())).cumsum()

    r0_table = r0_result.copy()
    r0_table["Time_dt"] = pd.to_datetime(r0_table["Time"], utc=True, errors="coerce")

    def is_pulse_state(s):
        return str(s).startswith(("CHA", "DCH"))

    def simulate_voltage(param, t_s, current_a, ocv_v, r0_ohm):
        r1_ohm, r2_ohm, tau1_s, tau2_s = param

        v1 = np.zeros(len(t_s), dtype=float)
        v2 = np.zeros(len(t_s), dtype=float)

        for k in range(1, len(t_s)):
            dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
            a1 = np.exp(-dt / tau1_s)
            a2 = np.exp(-dt / tau2_s)

            # 用当前采样点电流代表上一个采样间隔内的阶跃后电流
            # v[k] = 旧状态保留比例 * 旧极化电压 + 新状态建立比例 * 新稳态极化电压
            v1[k] = a1 * v1[k - 1] + r1_ohm * (1.0 - a1) * current_a[k]
            v2[k] = a2 * v2[k - 1] + r2_ohm * (1.0 - a2) * current_a[k]

        voltage_hat = ocv_v + current_a * r0_ohm + v1 + v2
        return voltage_hat

    result_rows = []

    for _, row in r0_table.iterrows():
        file_name = row["File"]
        pulse_time = row["Time_dt"]
        r0_ohm = float(row["R0"]) / 1000.0

        file_seq = seq[seq["File"].eq(file_name)].copy()
        if file_seq.empty:
            continue

        # 找到 r0_result 对应的 pulse 采样点所在片段
        pulse_candidates = file_seq[(file_seq["Time_dt"].eq(pulse_time)) & (file_seq["Zustand"].map(is_pulse_state))]

        if pulse_candidates.empty:
                continue

        pulse_segment_id = pulse_candidates.iloc[0]["segment_id"]

        pulse_df = file_seq[(file_seq["segment_id"].eq(pulse_segment_id)) & (file_seq["Zustand"].map(is_pulse_state))].copy()
        pulse_df = pulse_df.sort_values("Time_dt").reset_index(drop=True)

        if len(pulse_df) < 5:
            continue

        pulse_start_time = pulse_df["Time_dt"].iloc[0]
        pulse_end_time = pulse_df["Time_dt"].iloc[-1]

        prev_pauo = file_seq[(file_seq["Time_dt"] < pulse_start_time) & (file_seq["Zustand_clean"].eq("PAUO"))].copy()

        if prev_pauo.empty:
            continue

        prev_pauo_point = prev_pauo.sort_values("Time_dt").iloc[[-1]].copy()
        next_pauo_candidates = file_seq[(file_seq["Time_dt"].gt(pulse_end_time)) & (file_seq["Zustand_clean"].eq("PAUO"))].copy()

        if next_pauo_candidates.empty:
            continue

        next_pauo_segment_id = next_pauo_candidates.sort_values("Time_dt").iloc[0]["segment_id"]
        next_pauo_df = file_seq[(file_seq["segment_id"].eq(next_pauo_segment_id)) & (file_seq["Zustand_clean"].eq("PAUO"))].copy()
        next_pauo_df = next_pauo_df.sort_values("Time_dt").reset_index(drop=True)

        if len(next_pauo_df) < 5:
            continue
        
        # 脉冲前 PAUO 1 个点 + 脉冲过程所有点 + 脉冲后 PAUO 段所有点（直至下一个脉冲）
        fit_df = pd.concat(
            [prev_pauo_point, pulse_df, next_pauo_df],
            ignore_index=True,
            sort=False,
        ).sort_values("Time_dt").reset_index(drop=True)

        t_s = (fit_df["Time_dt"] - fit_df["Time_dt"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)

        current_a = fit_df["Current"].to_numpy(dtype=float)
        voltage_v = fit_df["Voltage"].to_numpy(dtype=float)

        ocv_before = float(prev_pauo_point["Voltage"].iloc[0])
        ocv_after = float(next_pauo_df["Voltage"].iloc[-1])

        # 从拟合窗口开始，到脉冲结束，一共过了多少秒
        pulse_end_s = (pulse_end_time - fit_df["Time_dt"].iloc[0]).total_seconds()
        pulse_end_s = max(float(pulse_end_s), 1e-9)

        # pulse 期间 OCV 从前 PAUO 末端近似过渡到后 PAUO 末端；pause 期间 OCV 近似保持为后 PAUO 末端
        ocv_v = np.where(
            t_s <= pulse_end_s,
            ocv_before + (ocv_after - ocv_before) * (t_s / pulse_end_s),
            ocv_after,
        )

        pulse_mask = fit_df["Zustand"].map(is_pulse_state).to_numpy(dtype=bool)
        pauo_mask = fit_df["Zustand_clean"].eq("PAUO").to_numpy(dtype=bool)
        prev_point_mask = fit_df["Time_dt"].eq(prev_pauo_point["Time_dt"].iloc[0]).to_numpy(dtype=bool)

        # 前一个 PAUO 末端点只用于初始化，不参与误差计算
        fit_mask = ~prev_point_mask
        pulse_fit_mask = fit_mask & pulse_mask
        pauo_fit_mask = fit_mask & pauo_mask

        n_pulse = int(pulse_fit_mask.sum())
        n_pauo = int(pauo_fit_mask.sum())

        weights = np.zeros(len(fit_df), dtype=float)
        # 设置 pulse段和pauo段在拟合中的权重占比
        pulse_weight_total = 0.7
        pauo_weight_total = 0.3
        if n_pulse > 0:
            weights[pulse_fit_mask] = np.sqrt(pulse_weight_total / n_pulse)
        if n_pauo > 0:
            weights[pauo_fit_mask] = np.sqrt(pauo_weight_total / n_pauo)

        if n_pulse == 0 or n_pauo == 0:
            continue

        # 初值：先用 pulse 末端电压粗略估计动态电阻，再分给两个 RC 支路
        i_pulse_mean = float(np.nanmedian(np.abs(pulse_df["Current"].to_numpy(dtype=float))))
        v_pulse_tail = float(np.nanmedian(pulse_df["Voltage"].tail(min(5, len(pulse_df))).to_numpy(dtype=float)))
        i_pulse_abs = float(np.nanmedian(np.abs(pulse_df["Current"].to_numpy(dtype=float))))

        if not np.isfinite(i_pulse_abs) or i_pulse_abs <= 1e-9:
            continue

        r_dyn_guess = abs((ocv_after - v_pulse_tail) / i_pulse_mean) - r0_ohm
        r_dyn_guess = float(np.clip(r_dyn_guess, 0.002, 0.08))

        x0 = np.array([
            0.4 * r_dyn_guess,   # R1, Ohm
            0.6 * r_dyn_guess,   # R2, Ohm
            5.0,                 # tau1, s
            800.0,               # tau2, s
        ], dtype=float)

        lower = np.array([1e-6, 1e-6, 0.05, 100.0], dtype=float)
        upper = np.array([0.1,  2.0,  100.0, 5000.0], dtype=float)
        x0 = np.minimum(np.maximum(x0, lower * 1.01), upper * 0.99)

        def residual(param):
            voltage_hat = simulate_voltage(param, t_s, current_a, ocv_v, r0_ohm)
            return (voltage_hat - voltage_v) * weights

        try:
            fit = least_squares(
                residual,
                x0=x0,
                bounds=(lower, upper),
                max_nfev=3000,
            )

            r1_ohm, r2_ohm, tau1_s, tau2_s = fit.x
            voltage_fit = simulate_voltage(fit.x, t_s, current_a, ocv_v, r0_ohm)

            err_mv = (voltage_fit[fit_mask] - voltage_v[fit_mask]) * 1000.0
            rmse_mv = float(np.sqrt(np.mean(err_mv ** 2)))

            c1_f = tau1_s / r1_ohm
            c2_f = tau2_s / r2_ohm

            output_row = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            output_row.update({
                "R1": r1_ohm * 1000.0,
                "R2": r2_ohm * 1000.0,
                "tau1": tau1_s,
                "tau2": tau2_s,
                "C1": c1_f,
                "C2": c2_f,
                "RMSE_mV": rmse_mv,
            })
            result_rows.append(output_row)

        except Exception:
            output_row = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            output_row.update({
                "R1": np.nan,
                "R2": np.nan,
                "tau1": np.nan,
                "tau2": np.nan,
                "C1": np.nan,
                "C2": np.nan,
                "RMSE_mV": np.nan,
            })
            result_rows.append(output_row)

    return pd.DataFrame(result_rows)


r1r2_result = resistance_r1r2(time_diff_sequence, r0_result)
display(r1r2_result)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R1,R2,tau1,tau2,C1,C2,RMSE_mV
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,4.121693,CHA,12_21,CHA/1.5,22.560928,9.093882,43.650883,2.580510,100.000023,283.763336,2290.904902,1.411745
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,4.017813,DCH,12_25,DCH/-3.0,23.507635,12.468150,111.673152,5.365663,670.262466,430.349552,6002.001840,1.945718
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,3.804263,CHA,12_39,CHA/1.5,18.318566,10.156197,140.270110,9.140925,891.935637,900.034241,6358.700654,1.591725
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.020000+00:00,-2.997524,3.721058,DCH,12_43,DCH/-3.0,18.856021,10.171664,246.439997,10.724958,1074.985610,1054.395552,4362.058202,2.570134
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.330000+00:00,2.992477,3.833616,CHA,12_47,CHA/3.0,18.242787,7.227011,20.325338,6.233315,100.038060,862.502424,4921.840050,0.396510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.320000+00:00,-2.998784,3.718723,DCH,9_44,DCH/-3.0,18.329025,9.938773,247.752879,10.694758,1087.404383,1076.064192,4389.068601,2.538533
190,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,3.829836,CHA,9_48,CHA/3.0,18.239417,7.531732,19.231355,7.897953,116.208876,1048.623655,6042.677435,0.324880
191,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,3.331177,CHA,9_58,CHA/1.5,20.284600,5.451521,1210.451218,6.259058,1290.775948,1148.130659,1066.359328,7.221923
192,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,3.244738,DCH,9_62,DCH/-3.0,20.406411,5.456888,1974.146950,7.642258,1388.758264,1400.479054,703.472588,15.734711
